# Tokenize transcriptomes and train PerturbGen

This notebook starts with the CPU-only tokenization step. Replace placeholder paths like `path/to/...` with real locations on your system.


## 1. Configure paths and parameters

These parameters mirror the CLI options in `perturbgen.pp.GF_tokenisation`.


In [32]:
H5AD_PATH = "/nfs/team361/am74/Cytomeister/Evaluation_datasets/LPS/lps_otar.h5ad" #"path/to/data.h5ad"
DATASET_NAME = "LPS_all_tps_2k"
GENE_FILTERING_MODE = "hvg"  # one of: hvg, degs, all
HVG_MODE = "before_tokenisation"  # before_tokenisation or after_tokenisation

VAR_LIST = [
    "cell_type_harmonized",
    "time_after_LPS",
]

PAIRING_MODE = "stratified"  # stratified, random, mapping
TIME_OBS = "time_after_LPS"
PAIRING_FILE = "path/to/pairing.csv"  # only if PAIRING_MODE == 'mapping'
MAIN_PAIRING_OBS = "cell_type_harmonized"
OPT_PAIRING_OBS = []  # optional additional obs

NPROC = 8
N_HVG = 2000
TIME_POINT_ORDER = ["normal", "90m_LPS", "6h_LPS", "10h_LPS"]
REFERENCE_TIME = "normal"

GENE_MEDIAN_PATH = "Perturbgen/perturbgen/pp/gene_median_dict_gftokens_gc95M.pkl"
TOKEN_DICT_PATH  = "Perturbgen/perturbgen/pp/token_dict_gftokens_gc95M.pkl"
GENE_MAPPING_PATH = "Perturbgen/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl"



## 2. Build the tokenization command

This prints the exact command that will be executed. Review it before running.


In [33]:
cmd = [
    "python",
    "-m",
    "perturbgen",
    "tokenise",
    "--h5ad_path", H5AD_PATH,
    "--dataset", DATASET_NAME,
    "--gene_filtering_mode", GENE_FILTERING_MODE,
    "--hvg_mode", HVG_MODE,
    "--var_list", *VAR_LIST,
    "--pairing_mode", PAIRING_MODE,
    "--time_obs", TIME_OBS,
    "--main_pairing_obs", MAIN_PAIRING_OBS,
    "--nproc", str(NPROC),
    "--n_hvg", str(N_HVG),
    "--reference_time", REFERENCE_TIME,
    "--time_point_order", *TIME_POINT_ORDER,
    "--gene_median_path", GENE_MEDIAN_PATH,
    "--token_dict_path", TOKEN_DICT_PATH,
    "--gene_mapping_path", GENE_MAPPING_PATH,
]

if PAIRING_MODE == "mapping":
    cmd += ["--pairing_file", PAIRING_FILE]
if OPT_PAIRING_OBS:
    cmd += ["--opt_pairing_obs", *OPT_PAIRING_OBS]

print(" ".join(cmd))


python -m perturbgen tokenise --h5ad_path /nfs/team361/am74/Cytomeister/Evaluation_datasets/LPS/lps_otar.h5ad --dataset LPS_all_tps_2k --gene_filtering_mode hvg --hvg_mode before_tokenisation --var_list cell_type_harmonized time_after_LPS --pairing_mode stratified --time_obs time_after_LPS --main_pairing_obs cell_type_harmonized --nproc 8 --n_hvg 2000 --reference_time normal --time_point_order normal 90m_LPS 6h_LPS 10h_LPS --gene_median_path Perturbgen/perturbgen/pp/gene_median_dict_gftokens_gc95M.pkl --token_dict_path Perturbgen/perturbgen/pp/token_dict_gftokens_gc95M.pkl --gene_mapping_path Perturbgen/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl


## 3. Run tokenization (CPU-only)

This step can take time depending on dataset size.


In [ ]:
import subprocess

subprocess.run(cmd, check=True)


loading, please wait...
Current working directory: /lustre/scratch126/cellgen/lotfollahi/dv8/for_otar
Start preprocessing adata...
Number of genes dropped: 0
Finished preprocessing adata.
Start tokenisation of adata...
Tokenizing /lustre/scratch126/cellgen/lotfollahi/dv8/for_otar/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg/LPS_all_tps_2k.h5ad


/lustre/scratch126/cellgen/lotfollahi/dv8/for_otar/Perturbgen/perturbgen/pp/tokenizer.py:495: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/lustre/scratch126/cellgen/lotfollahi/dv8/for_otar/Perturbgen/perturbgen/pp/tokenizer.py:498: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/lustre/scratch126/cellgen/lotfollahi/dv8/for_otar/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg/LPS_all_tps_2k.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.


Saving the dataset (1/1 shards): 100%|██████████| 223478/223478 [00:00<00:00, 458199.76 examples/s]


Finished tokenisation.


/lustre/scratch126/cellgen/lotfollahi/dv8/for_otar/Perturbgen/perturbgen/src/utils.py:1643: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata_obs_.groupby(grouping_obs)[time_obs].transform('nunique') == total_tps
/lustre/scratch126/cellgen/lotfollahi/dv8/for_otar/Perturbgen/perturbgen/src/utils.py:1646: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = adata_grouped.groupby(grouping_obs)
  0%|          | 0/4 [00:00<?, ?it/s]/lustre/scratch126/cellgen/lotfollahi/dv8/for_otar/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str in

## 4. Outputs

Tokenized files are written under the tokenized data directory defined in `perturbgen/configs/paths.py`.
You should see a new folder for your dataset name containing `.dataset` files and pairing outputs in root_dir/T_perturb/tokenized_data.


---
Next: training steps (masking model and count decoder) in the following sections.


## 5. Train masking model (GPU required)

This step requires a GPU. Run on a GPU node (e.g., via your scheduler) or a machine with CUDA.


In [ ]:
# GPU required
OUTPUT_DIR = "T_perturb/res/masking"  # relative to repo root
SRC_DATASET = "T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset" # path/to/src_dataset.dataset from tokenization step
TGT_DATASET_FOLDER = "T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt" # path/to/tgt_datasets_folder from tokenization step
SRC_ADATA = "T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad" # path/to/src_adata.h5ad" from tokenization step
TGT_ADATA_FOLDER = "T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt" # path/to/tgt_adata_folder from tokenization step
MAPPING_DICT_PATH = "T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl" # path/to/mapping_dict.pkl from tokenization step

ENCODER_PATH = "Perturbgen/pretraining_cohort/20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt"

BATCH_SIZE = 64
MAX_LEN = 797
EPOCHS = 20
TGT_VOCAB_SIZE = 2002
CELLGEN_LR = 1e-4
CELLGEN_WD = 1e-4
N_WORKERS = 4
NUM_LAYERS = 6
D_FF = 32
PRED_TPS = ["1", "2", "3"]

VAR_LIST = ["cell_type_harmonized", "time_after_LPS"]
COND_LIST = ["cell_type_harmonized"]

SEED = 0
CONTEXT_MODE = "True"
POS_ENCODING_MODE = "time_pos_sin"
MASK_SCHEDULER = "pow"
NUM_NODE = 1
D_MODEL = 768

CKPT_MASKING_PATH = "path/to/optional_resume.ckpt"  # optional
USE_WEIGHTED_SAMPLER = "False"


In [2]:
cmd = [
    "python",
    "-m",
    "perturbgen",
    "train-mask",
    "--train_mode", "masking",
    "--split", "False",
    "--encoder", "scmaskgit",
    "--splitting_mode", "stratified",
    "--split_obs", "cell_type_harmonized",
    "--output_dir", OUTPUT_DIR,
    "--src_dataset", SRC_DATASET,
    "--tgt_dataset_folder", TGT_DATASET_FOLDER,
    "--src_adata", SRC_ADATA,
    "--tgt_adata_folder", TGT_ADATA_FOLDER,
    "--mapping_dict_path", MAPPING_DICT_PATH,
    "--batch_size", str(BATCH_SIZE),
    "--max_len", str(MAX_LEN),
    "--epochs", str(EPOCHS),
    "--tgt_vocab_size", str(TGT_VOCAB_SIZE),
    "--cellgen_lr", str(CELLGEN_LR),
    "--cellgen_wd", str(CELLGEN_WD),
    "--n_workers", str(N_WORKERS),
    "--num_layers", str(NUM_LAYERS),
    "--d_ff", str(D_FF),
    "--pred_tps", *PRED_TPS,
    "--var_list", *VAR_LIST,
    "--cond_list", *COND_LIST,
    "--encoder_path", ENCODER_PATH,
    "--seed", str(SEED),
    "--context_mode", CONTEXT_MODE,
    "--pos_encoding_mode", POS_ENCODING_MODE,
    "--mask_scheduler", MASK_SCHEDULER,
    "--num_node", str(NUM_NODE),
    "--d_model", str(D_MODEL),
    "--use_weighted_sampler", USE_WEIGHTED_SAMPLER,
]

if CKPT_MASKING_PATH:
    cmd += ["--ckpt_masking_path", CKPT_MASKING_PATH]

print(" ".join(cmd))


python -m perturbgen train-mask --train_mode masking --split False --encoder scmaskgit --splitting_mode stratified --split_obs cell_type_harmonized --output_dir T_perturb/res/masking --src_dataset T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset --tgt_dataset_folder T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt --src_adata T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad --tgt_adata_folder T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt --mapping_dict_path T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl --batch_size 64 --max_len 797 --epochs 20 --tgt_vocab_size 2002 --cellgen_lr 0.0001 --cellgen_wd 0.0001 --n_workers 4 --num_layers 6 --d_ff 32 --pred_tps 1 2 3 --var_list cell_type_harmonized time_after_LPS --cond_list time_after_LPS --encoder_path Perturbgen/pretraining_cohort/20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-

In [ ]:
import subprocess

subprocess.run(cmd, check=True)


## 6. Train count decoder (GPU required)

This step trains the count decoder using the masking checkpoint.


In [1]:
# GPU required
COUNT_OUTPUT_DIR = "T_perturb/res/count"  # relative to repo root
CKPT_MASKING_PATH = "path/to/masking_checkpoint.ckpt" # should be selected based on best masking model from previous step

SRC_DATASET = "T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset" # path/to/src_dataset.dataset from tokenization step
TGT_DATASET_FOLDER = "T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt" # path/to/tgt_datasets_folder from tokenization step
SRC_ADATA = "T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad" # path/to/src_adata.h5ad" from tokenization step
TGT_ADATA_FOLDER = "T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt"  # path/to/tgt_adata_folder from tokenization step
MAPPING_DICT_PATH = "T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl" # path/to/mapping_dict.pkl from tokenization step

ENCODER_PATH = "Perturbgen/pretraining_cohort/20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt"

BATCH_SIZE = 16
MAX_LEN = 797
EPOCHS = 16
TGT_VOCAB_SIZE = 2002
COUNT_LR = 0.001
CELLGEN_LR = 0.0001
CELLGEN_WD = 0.0001
COUNT_WD = 0.001
MLM_PROB = 0.30
N_WORKERS = 32
NUM_LAYERS = 6
D_FF = 32
LOSS_MODE = "zinb"
PRED_TPS = ["1", "2", "3"]

VAR_LIST = ["cell_type_harmonized", "time_after_LPS"]
COND_LIST = ["cell_type_harmonized"]

ADD_CELL_TIME = "False"
D_CONDC = 64
D_CONDT = 768
COUNT_DROPOUT = 0.1
USE_POSITIONAL_ENCODING = "False"
LAYER_NORM = "True"
CONTEXT_MODE = "True"
POS_ENCODING_MODE = "time_pos_sin"
MASK_SCHEDULER = "cosine"
NUM_NODE = 1
D_MODEL = 768


In [2]:
cmd = [
    "python",
    "-m",
    "perturbgen",
    "train-decoder",
    "--train_mode", "count",
    "--split", "False",
    "--splitting_mode", "stratified",
    "--output_dir", COUNT_OUTPUT_DIR,
    "--ckpt_masking_path", CKPT_MASKING_PATH,
    "--src_dataset", SRC_DATASET,
    "--tgt_dataset_folder", TGT_DATASET_FOLDER,
    "--src_adata", SRC_ADATA,
    "--tgt_adata_folder", TGT_ADATA_FOLDER,
    "--mapping_dict_path", MAPPING_DICT_PATH,
    "--batch_size", str(BATCH_SIZE),
    "--max_len", str(MAX_LEN),
    "--epochs", str(EPOCHS),
    "--tgt_vocab_size", str(TGT_VOCAB_SIZE),
    "--count_lr", str(COUNT_LR),
    "--cellgen_lr", str(CELLGEN_LR),
    "--cellgen_wd", str(CELLGEN_WD),
    "--count_wd", str(COUNT_WD),
    "--mlm_prob", str(MLM_PROB),
    "--n_workers", str(N_WORKERS),
    "--num_layers", str(NUM_LAYERS),
    "--d_ff", str(D_FF),
    "--loss_mode", LOSS_MODE,
    "--pred_tps", *PRED_TPS,
    "--var_list", *VAR_LIST,
    "--cond_list", *COND_LIST,
    "--encoder", "scmaskgit",
    "--add_cell_time", ADD_CELL_TIME,
    "--d_condc", str(D_CONDC),
    "--d_condt", str(D_CONDT),
    "--count_dropout", str(COUNT_DROPOUT),
    "--use_positional_encoding", USE_POSITIONAL_ENCODING,
    "--layer_norm", LAYER_NORM,
    "--context_mode", CONTEXT_MODE,
    "--encoder_path", ENCODER_PATH,
    "--pos_encoding_mode", POS_ENCODING_MODE,
    "--mask_scheduler", MASK_SCHEDULER,
    "--num_node", str(NUM_NODE),
    "--d_model", str(D_MODEL),
]

print(" ".join(cmd))


python -m perturbgen train-decoder --train_mode count --split False --splitting_mode stratified --output_dir T_perturb/res/count --ckpt_masking_path path/to/masking_checkpoint.ckpt --src_dataset T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset --tgt_dataset_folder T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt --src_adata T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad --tgt_adata_folder T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt --mapping_dict_path T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl --batch_size 16 --max_len 797 --epochs 16 --tgt_vocab_size 2002 --count_lr 0.001 --cellgen_lr 0.0001 --cellgen_wd 0.0001 --count_wd 0.001 --mlm_prob 0.3 --n_workers 32 --num_layers 6 --d_ff 32 --loss_mode zinb --pred_tps 1 2 3 --var_list cell_type_harmonized time_after_LPS --cond_list cell_type_harmonized --encoder scmaskgit --add_cell_time False --d_condc 64 --d_condt 768 --

In [ ]:
import subprocess

subprocess.run(cmd, check=True)


# 7. Perturbation (in silico)

This notebook runs in silico perturbation using a count-decoder checkpoint and a YAML config.
Update paths in the config before running.


## 7-1. Config

Edit the config file:
`docs/examples/configs/perturbation.yaml`


## 7-2. Parameters (GPU required)

This step requires a GPU. Update the config path if needed.


## 

This uses the repository script at `perturbgen/Perturb/val.py`.


## 7-3. Perturbation (GPU required)

Run in silico perturbation using `val.py` and a YAML config.


In [ ]:
VAL_SCRIPT = "perturbgen/Perturb/val.py"
CONFIG_PATH = "docs/examples/configs/perturbation.yaml"


In [ ]:
cmd = [
    "python", VAL_SCRIPT,
    "--config", CONFIG_PATH,
]

print(" ".join(cmd))


In [ ]:
import subprocess

subprocess.run(cmd, check=True)
